In [0]:
%python 
import pandas as pd
df_students = pd.read_csv("/Volumes/workspace/default/python_ex/school_student_performance.csv")
df_students.info()
df_students.head()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   student_id       15 non-null     int64  
 1   student_name     15 non-null     object 
 2   grade            15 non-null     int64  
 3   class_group      15 non-null     object 
 4   gender           15 non-null     object 
 5   math_score       15 non-null     int64  
 6   english_score    14 non-null     float64
 7   science_score    15 non-null     int64  
 8   attendance_rate  14 non-null     float64
 9   city             13 non-null     object 
dtypes: float64(2), int64(4), object(4)
memory usage: 1.3+ KB


,student_id,student_name,grade,class_group,gender,math_score,english_score,science_score,attendance_rate,city
0,1001,Amina,10,10A,Female,78,82.0,74,0.92,Johannesburg
1,1002,Thabo,10,10B,Male,64,70.0,68,0.85,Pretoria
2,1003,Lerato,11,11A,Female,88,91.0,86,0.96,Soweto
3,1004,John,11,11B,Male,55,60.0,58,0.78,Durban
4,1005,Zanele,12,12A,Female,92,89.0,94,0.98,Cape Town


In [0]:
df_students.columns = [col.lower() for col in df_students.columns]
df_students['city'] = df_students['city'].fillna('Unknown')
df_students['english_score'] = df_students['english_score'].fillna(df_students['english_score'].mean())
df_students['attendance_rate'] = df_students['attendance_rate'].fillna(df_students['attendance_rate'].mean())
df_students['student_name'] = df_students['student_name'].str.title().str.strip()



In [0]:
df_students['score']=(df_students['math_score']+df_students['english_score']+df_students['science_score'])/3
df_students['result']=df_students['score'].apply(lambda score: 'Pass' if score>=50 else 'Fail')

In [0]:
def attendance(attendance_rate):
    if attendance_rate >=0.9:
        return 'High Attendance'
    if attendance_rate>=0.75:
        return 'Low Attendance'
    else:
        return 'At Risk'
df_students['attendance_category'] = df_students['attendance_rate'].apply(attendance)

def performance_category(score):
    if score>=80:
        return 'Excellent'
    if score>=60:
        return 'Good'
    if score>=50:
        return 'Average'
    else:
        return 'Poor'
df_students['performance_category'] = df_students['score'].apply(performance_category)

df_students.head()



,student_id,student_name,grade,class_group,gender,math_score,english_score,science_score,attendance_rate,city,score,result,attendance_category,performance_category
0,1001,Amina,10,10A,Female,78,82.0,74,0.92,Johannesburg,78.000000,Pass,High Attendance,Good
1,1002,Thabo,10,10B,Male,64,70.0,68,0.85,Pretoria,67.333333,Pass,Low Attendance,Good
2,1003,Lerato,11,11A,Female,88,91.0,86,0.96,Soweto,88.333333,Pass,High Attendance,Excellent
3,1004,John,11,11B,Male,55,60.0,58,0.78,Durban,57.666667,Pass,Low Attendance,Average
4,1005,Zanele,12,12A,Female,92,89.0,94,0.98,Cape Town,91.666667,Pass,High Attendance,Excellent


In [0]:
df_lower_attandence = df_students[df_students['attendance_category']=='Low Attendance']
df_lower_attandence.head()

,student_id,student_name,grade,class_group,gender,math_score,english_score,science_score,attendance_rate,city,score,result,attendance_category,performance_category
1,1002,Thabo,10,10B,Male,64,70.0,68,0.850,Pretoria,67.333333,Pass,Low Attendance,Good
3,1004,John,11,11B,Male,55,60.0,58,0.780,Durban,57.666667,Pass,Low Attendance,Average
7,1008,Sipho,11,11A,Male,72,67.0,70,0.855,Pretoria,69.666667,Pass,Low Attendance,Good
8,1009,Karabo,12,12A,Female,69,73.0,71,0.880,Midrand,71.000000,Pass,Low Attendance,Good
13,1014,Daniel,11,11A,Male,66,63.0,65,0.820,Unknown,64.666667,Pass,Low Attendance,Good


In [0]:
spark_df = spark.createDataFrame(df_students)
spark_df.createTempView('school_performance_updated')

In [0]:
%sql 
SELECT COUNT(student_id) AS Students,
attendance_category
FROM school_performance_updated
GROUP BY attendance_category ;


Students,attendance_category
7,High Attendance
5,Low Attendance
3,At Risk


In [0]:
%sql
SELECT *
FROM school_performance_updated;

student_id,student_name,grade,class_group,gender,math_score,english_score,science_score,attendance_rate,city,score,result,attendance_category,performance_category
1001,Amina,10,10A,Female,78,82.0,74,0.92,Johannesburg,78.0,Pass,High Attendance,Good
1002,Thabo,10,10B,Male,64,70.0,68,0.85,Pretoria,67.33333333333333,Pass,Low Attendance,Good
1003,Lerato,11,11A,Female,88,91.0,86,0.96,Soweto,88.33333333333333,Pass,High Attendance,Excellent
1004,John,11,11B,Male,55,60.0,58,0.78,Durban,57.666666666666664,Pass,Low Attendance,Average
1005,Zanele,12,12A,Female,92,89.0,94,0.98,Cape Town,91.66666666666667,Pass,High Attendance,Excellent
1006,Neo,12,12B,Male,47,54.0,50,0.69,Johannesburg,50.333333333333336,Pass,At Risk,Average
1007,Fatima,10,10A,Female,81,84.0,79,0.9,Unknown,81.33333333333333,Pass,High Attendance,Excellent
1008,Sipho,11,11A,Male,72,67.0,70,0.8550000000000001,Pretoria,69.66666666666667,Pass,Low Attendance,Good
1009,Karabo,12,12A,Female,69,73.0,71,0.88,Midrand,71.0,Pass,Low Attendance,Good
1010,Musa,10,10B,Male,35,42.0,39,0.62,Soweto,38.666666666666664,Fail,At Risk,Poor


In [0]:
%sql
SELECT grade , ROUND(AVG(score), 2) AS average_score
FROM school_performance_updated
GROUP BY  grade
ORDER BY grade DESC;

grade,average_score
12,73.79
11,71.47
10,70.27


In [0]:
%sql
SELECT student_id, student_name, attendance_category
FROM school_performance_updated
WHERE attendance_category = 'Low Attendance' OR attendance_category = 'At Risk'
GROUP BY student_id, student_name, attendance_category;
    


student_id,student_name,attendance_category
1002,Thabo,Low Attendance
1004,John,Low Attendance
1006,Neo,At Risk
1008,Sipho,Low Attendance
1009,Karabo,Low Attendance
1010,Musa,At Risk
1012,Brian,At Risk
1014,Daniel,Low Attendance


In [0]:
df_students.to_csv("/Volumes/workspace/default/python_ex/school_student_performance_updated.csv", index=False)
df_students.to_parquet("/Volumes/workspace/default/python_ex/school_student_performance_updated.parquet")